# 06 — Teste Estatístico

Treina cada modelo (U-Net 3D, Swin UNETR, SegResNet) $n$ vezes com seus melhores hiperparâmetros e compara os resultados com testes estatísticos.

**Fluxo:**
1. Definição dos modelos e hiperparâmetros
2. Treinamento repetido (cada modelo $n$ vezes)
3. Coleta de métricas (Dice, HD95, Sensitivity, Specificity — por classe)
4. Análise estatística entre e intra-modelos

In [ ]:
import os
import json
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from scipy import stats
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    CropForegroundd, NormalizeIntensityd, RandCropByPosNegLabeld,
    RandFlipd, ToTensord, MapLabelValued, AsDiscrete,
)
from monai.data import Dataset, DataLoader, decollate_batch
from monai.metrics import DiceMetric, HausdorffDistanceMetric, ConfusionMatrixMetric
from monai.networks.nets import UNet, SwinUNETR, SegResNet
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference

SEED_BASE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_DIR = Path('../results/statistical_test')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuração

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────
# Número de execuções por modelo (mínimo 5 recomendado para estatística)
N_RUNS = 5

# Épocas por execução
N_EPOCHS = 100

# Validar a cada N épocas
VAL_INTERVAL = 4

# Caminho do dataset (sample 1% para teste rápido (é oq tem pra hoje fio);
DATA_PATH = '../data/processed/MICCAI_BraTS2020_TrainingData_sample_1pct'
# DATA_PATH = '../data/raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

# ROI size para inferência deslizante (96 = mesmo do treinamento)
SW_ROI_SIZE = (96, 96, 96)
SW_BATCH_SIZE = 4

print(f'Execuções por modelo: {N_RUNS}')
print(f'Épocas: {N_EPOCHS}')
print(f'Dataset: {DATA_PATH}')

## 2. Pipeline de dados

In [ ]:
def get_data_dicts(data_path):
    patient_folders = sorted(
        glob.glob(os.path.join(data_path, 'BraTS20_Training_*'))
    )
    if len(patient_folders) == 0:
        raise ValueError(f'Nenhum paciente encontrado em: {data_path}')
    data_dicts = []
    for folder in patient_folders:
        pid = os.path.basename(folder)
        imgs = [os.path.join(folder, f'{pid}_{m}.nii') for m in ['flair','t1','t1ce','t2']]
        seg = os.path.join(folder, f'{pid}_seg.nii')
        if all(os.path.exists(f) for f in imgs) and os.path.exists(seg):
            data_dicts.append({'image': imgs, 'label': seg})
    print(f'Pacientes carregados: {len(data_dicts)}')
    return data_dicts


def make_transforms():
    keys = ['image', 'label']
    train_tf = Compose([
        LoadImaged(keys=keys), EnsureChannelFirstd(keys=keys),
        MapLabelValued(keys=['label'], orig_labels=[4], target_labels=[3]),
        Spacingd(keys=keys, pixdim=(1.0,1.0,1.0), mode=('bilinear','nearest')),
        Orientationd(keys=keys, axcodes='RAS'),
        CropForegroundd(keys=keys, source_key='image'),
        NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
        RandCropByPosNegLabeld(keys=keys, label_key='label', spatial_size=(96,96,96),
                               pos=1, neg=1, num_samples=2, image_key='image', image_threshold=0),
        RandFlipd(keys=keys, spatial_axis=[0], prob=0.10),
        ToTensord(keys=keys),
    ])
    val_tf = Compose([
        LoadImaged(keys=keys), EnsureChannelFirstd(keys=keys),
        MapLabelValued(keys=['label'], orig_labels=[4], target_labels=[3]),
        Spacingd(keys=keys, pixdim=(1.0,1.0,1.0), mode=('bilinear','nearest')),
        Orientationd(keys=keys, axcodes='RAS'),
        CropForegroundd(keys=keys, source_key='image'),
        NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
        ToTensord(keys=keys),
    ])
    return train_tf, val_tf


import glob

data_dicts = get_data_dicts(DATA_PATH)
train_files, val_files = data_dicts[:-max(1, round(len(data_dicts)*0.11))], data_dicts[-max(1, round(len(data_dicts)*0.11)):]
print(f'Treino: {len(train_files)} | Validação: {len(val_files)}')

## 3. Definição dos modelos e hiperparâmetros

In [ ]:
MODEL_CONFIGS = {
    'UNet3D': {
        'model_fn': lambda: UNet(
            spatial_dims=3, in_channels=4, out_channels=4,
            channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
            num_res_units=2,
        ),
        'lr': 2.3237e-4,
        'optimizer': 'Adam',
        'weight_decay': 0.0,
    },
    'SwinUNETR': {
        'model_fn': lambda: SwinUNETR(
            in_channels=4, out_channels=4,
            feature_size=12, use_checkpoint=True,
        ),
        'lr': 7.3324e-4,
        'optimizer': 'AdamW',
        'weight_decay': 2.768e-5,
    },
    'SegResNet': {
        'model_fn': lambda: SegResNet(
            in_channels=4, out_channels=4,
            init_filters=32, dropout_prob=0.1848,
        ),
        'lr': 4.1678e-5,
        'optimizer': 'Adam',
        'weight_decay': 0.0,
    },
}

LOSS_FN = DiceCELoss(to_onehot_y=True, softmax=True)

for name, cfg in MODEL_CONFIGS.items():
    m = cfg['model_fn']()
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name}: {n_params:,} parâmetros | lr={cfg["lr"]:.2e} | opt={cfg["optimizer"]}')

## 4. Funções de treino e avaliação

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        imgs = batch['image'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, device, roi_size, sw_batch_size):
    model.eval()
    dice_metric = DiceMetric(include_background=False, reduction='mean')
    with torch.no_grad():
        for batch in loader:
            imgs = batch['image'].to(device)
            labels = batch['label'].to(device)
            outputs = sliding_window_inference(inputs=imgs, roi_size=roi_size,
                                               sw_batch_size=sw_batch_size, predictor=model)
            post_pred = AsDiscrete(argmax=True, to_onehot=4)
            post_label = AsDiscrete(to_onehot=4)
            outputs = [post_pred(i) for i in decollate_batch(outputs)]
            labels = [post_label(i) for i in decollate_batch(labels)]
            dice_metric(y_pred=outputs, y=labels)
    return dice_metric.aggregate().item()


def evaluate_detailed(model, loader, device, roi_size, sw_batch_size):
    """Avaliação detalhada: Dice, HD95, Sensitivity, Specificity por classe."""
    model.eval()
    class_names = ['NCR', 'ED', 'ET']  # classes 1, 2, 3 (sem background)

    dice_m = DiceMetric(include_background=False, reduction='mean_batch')
    hd95_m = HausdorffDistanceMetric(include_background=False, reduction='mean_batch',
                                      percentile=95, metric='euclidean')
    sens_m = ConfusionMatrixMetric(include_background=False, metric_name='sensitivity',
                                    reduction='mean_batch')
    spec_m = ConfusionMatrixMetric(include_background=False, metric_name='specificity',
                                    reduction='mean_batch')

    post_pred = AsDiscrete(argmax=True, to_onehot=4)
    post_label = AsDiscrete(to_onehot=4)

    with torch.no_grad():
        for batch in loader:
            imgs = batch['image'].to(device)
            labels = batch['label'].to(device)
            outputs = sliding_window_inference(inputs=imgs, roi_size=roi_size,
                                               sw_batch_size=sw_batch_size, predictor=model)
            outputs_d = [post_pred(i) for i in decollate_batch(outputs)]
            labels_d = [post_label(i) for i in decollate_batch(labels)]
            dice_m(y_pred=outputs_d, y=labels_d)
            hd95_m(y_pred=outputs_d, y=labels_d)
            sens_m(y_pred=outputs_d, y=labels_d)
            spec_m(y_pred=outputs_d, y=labels_d)

    results = {}
    for i, name in enumerate(class_names):
        results[name] = {
            'dice': dice_m.aggregate()[i].item(),
            'hd95': hd95_m.aggregate()[i].item(),
            'sensitivity': sens_m.aggregate()[i].item(),
            'specificity': spec_m.aggregate()[i].item(),
        }
    results['mean'] = {
        'dice': dice_m.aggregate().mean().item(),
        'hd95': hd95_m.aggregate().mean().item(),
        'sensitivity': sens_m.aggregate().mean().item(),
        'specificity': spec_m.aggregate().mean().item(),
    }
    return results

## 5. Loop de treinamento com $n$ execuções

In [ ]:
# Estrutura para armazenar todos os resultados:
# all_results[model_name][run_idx] = {'val_dice_per_epoch': [...], 'final_metrics': {...}}
all_results = {name: [] for name in MODEL_CONFIGS}

for model_name, cfg in MODEL_CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'Modelo: {model_name}')
    print(f'{"="*60}')

    for run in range(N_RUNS):
        seed = SEED_BASE + run
        set_seed(seed)
        print(f'\n--- Run {run+1}/{N_RUNS} (seed={seed}) ---')

        # Dataloaders frescos a cada run
        train_tf, val_tf = make_transforms()
        train_ds = Dataset(data=train_files, transform=train_tf)
        val_ds = Dataset(data=val_files, transform=val_tf)
        train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=4)
        val_loader = DataLoader(val_ds, batch_size=1, num_workers=4)

        # Modelo e otimizador frescos
        model = cfg['model_fn']().to(DEVICE)
        if cfg['optimizer'] == 'AdamW':
            optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                           weight_decay=cfg['weight_decay'])
        else:
            optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'],
                                         weight_decay=cfg['weight_decay'])

        # Treinamento
        val_dice_history = []
        best_dice = -1.0
        best_model_state = None

        for epoch in range(1, N_EPOCHS + 1):
            train_loss = train_epoch(model, train_loader, optimizer, LOSS_FN, DEVICE)
            if epoch % VAL_INTERVAL == 0:
                val_dice = validate(model, val_loader, DEVICE, SW_ROI_SIZE, SW_BATCH_SIZE)
                val_dice_history.append(val_dice)
                if val_dice > best_dice:
                    best_dice = val_dice
                    best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                print(f'  Epoch {epoch:3d} | loss={train_loss:.4f} | val_dice={val_dice:.4f} | best={best_dice:.4f}')

        # Avaliação detalhada com o melhor modelo
        model.load_state_dict(best_model_state)
        model = model.to(DEVICE)
        final_metrics = evaluate_detailed(model, val_loader, DEVICE, SW_ROI_SIZE, SW_BATCH_SIZE)
        print(f'  Run {run+1} final: mean_dice={final_metrics["mean"]["dice"]:.4f} '
              f'mean_hd95={final_metrics["mean"]["hd95"]:.4f}')

        run_result = {
            'seed': seed,
            'val_dice_per_epoch': val_dice_history,
            'best_dice_during_training': best_dice,
            'final_metrics': final_metrics,
        }
        all_results[model_name].append(run_result)

        # Salvar incrementalmente
        save_path = RESULTS_DIR / f'{model_name}_run{run}.json'
        with open(save_path, 'w') as f:
            json.dump(run_result, f, indent=2)

        # Liberar GPU
        del model, optimizer, best_model_state
        torch.cuda.empty_cache()

print('\nTreinamento concluído!')

## 6. Análise estatística

In [ ]:
# Se já existem resultados salvos, carregá-los em vez de re-treinar
import os
loaded_from_disk = False
for model_name in MODEL_CONFIGS:
    if not all_results[model_name]:
        runs = []
        i = 0
        while True:
            p = RESULTS_DIR / f'{model_name}_run{i}.json'
            if p.exists():
                with open(p) as f:
                    runs.append(json.load(f))
                i += 1
            else:
                break
        if runs:
            all_results[model_name] = runs
            loaded_from_disk = True

if loaded_from_disk:
    print('Resultados carregados do disco.')

for name, runs in all_results.items():
    print(f'{name}: {len(runs)} execuções')

### 6.1 Resumo descritivo

In [ ]:
model_names = list(MODEL_CONFIGS.keys())
metric_keys = ['dice', 'hd95', 'sensitivity', 'specificity']
class_labels = ['NCR', 'ED', 'ET']

# Coletar métricas por modelo e por run
summary = {}
for model_name in model_names:
    runs = all_results[model_name]
    summary[model_name] = {}
    for metric in metric_keys:
        for cls in class_labels + ['mean']:
            key = f'{cls}_{metric}'
            values = [r['final_metrics'][cls][metric] for r in runs]
            summary[model_name][key] = {
                'mean': np.mean(values),
                'std': np.std(values, ddof=1),
                'min': np.min(values),
                'max': np.max(values),
                'values': values,
            }

# Exibir tabela resumo
print(f'{"Modelo":<12} {"Métrica":<14} {"Mean":<8} {"Std":<8} {"Min":<8} {"Max":<8}')
print('-' * 58)
for model_name in model_names:
    for metric in metric_keys:
        key = f'mean_{metric}'
        s = summary[model_name][key]
        print(f'{model_name:<12} {metric:<14} {s["mean"]:<8.4f} {s["std"]:<8.4f} {s["min"]:<8.4f} {s["max"]:<8.4f}')
    print()

### 6.2 Curvas de aprendizado

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, model_name in zip(axes, model_names):
    runs = all_results[model_name]
    epochs = [VAL_INTERVAL * (i+1) for i in range(len(runs[0]['val_dice_per_epoch']))]

    for i, run in enumerate(runs):
        ax.plot(epochs, run['val_dice_per_epoch'], alpha=0.5,
                label=f'Run {i+1}' if len(runs) <= 8 else None)

    # Média das runs
    mean_curve = np.mean([r['val_dice_per_epoch'] for r in runs], axis=0)
    ax.plot(epochs, mean_curve, 'k-', linewidth=2, label='Média')

    ax.set_title(model_name)
    ax.set_xlabel('Época')
    ax.set_ylabel('Val Dice')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Box plots — Dice e HD95 por classe

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for col, cls in enumerate(class_labels + ['Mean']):
    # Dice
    dice_data = [summary[m][f'{cls}_dice']['values'] for m in model_names]
    axes[0][col].boxplot(dice_data, labels=[m.replace('3D','\n3D') for m in model_names])
    axes[0][col].set_title(f'Dice — {cls}')
    axes[0][col].grid(True, alpha=0.3)

    # HD95
    hd_data = [summary[m][f'{cls}_hd95']['values'] for m in model_names]
    axes[1][col].boxplot(hd_data, labels=[m.replace('3D','\n3D') for m in model_names])
    axes[1][col].set_title(f'HD95 — {cls}')
    axes[1][col].grid(True, alpha=0.3)

axes[0][0].set_ylabel('Dice Score')
axes[1][0].set_ylabel('HD95 (mm)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Barras com intervalo de confiança (95%)

In [ ]:
from math import sqrt

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for col, cls in enumerate(class_labels + ['Mean']):
    means, ci_lows, ci_highs = [], [], []
    for model_name in model_names:
        values = summary[model_name][f'{cls}_dice']['values']
        m = np.mean(values)
        s = np.std(values, ddof=1)
        n = len(values)
        # IC 95% via t-distribution
        ci = stats.t.ppf(0.975, df=n-1) * s / sqrt(n)
        means.append(m)
        ci_lows.append(m - ci)
        ci_highs.append(m + ci)

    yerr = [m - lo for m, lo in zip(means, ci_lows)]
    axes[col].bar(model_names, means, yerr=yerr, capsize=5,
                  color=['#4C72B0', '#55A868', '#C44E52'], alpha=0.85)
    axes[col].set_title(f'Dice — {cls}')
    axes[col].set_ylabel('Dice Score')
    axes[col].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bar_ci.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Testes estatísticos pareados

In [ ]:
from itertools import combinations

print('=' * 72)
print('Testes estatísticos pareados (Wilcoxon signed-rank)')
print('H0: não há diferença entre os modelos. H1: há diferença.')
print('=' * 72)

for cls in class_labels + ['mean']:
    for metric in ['dice', 'hd95']:
        print(f'\n--- {cls} — {metric.upper()} ---')
        for m1, m2 in combinations(model_names, 2):
            v1 = summary[m1][f'{cls}_{metric}']['values']
            v2 = summary[m2][f'{cls}_{metric}']['values']
            stat, p = stats.wilcoxon(v1, v2, alternative='two-sided')
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
            print(f'  {m1} vs {m2}: W={stat:.1f}, p={p:.4f} {sig}')

print('\n*** p<0.001  ** p<0.01  * p<0.05  n.s.=não significativo')

### 6.6 Estabilidade intra-modelo (coeficiente de variação)

In [ ]:
print('Coeficiente de Variação (CV = std/mean) — menor = mais estável\n')
print(f'{"Modelo":<12} {"Dice_CV":<12} {"HD95_CV":<12}')
print('-' * 36)
for model_name in model_names:
    dice_vals = summary[model_name]['mean_dice']['values']
    hd_vals = summary[model_name]['mean_hd95']['values']
    cv_dice = np.std(dice_vals, ddof=1) / np.mean(dice_vals) * 100
    cv_hd = np.std(hd_vals, ddof=1) / np.mean(hd_vals) * 100
    print(f'{model_name:<12} {cv_dice:<12.2f}% {cv_hd:<12.2f}%')

### 6.7 Tabela completa por classe

In [ ]:
print(f'{"Modelo":<12} {"Classe":<6} {"Métrica":<14} {"Mean":<8} {"±Std":<8} {"IC95%":<16}')
print('=' * 72)
for model_name in model_names:
    for cls in class_labels + ['Mean']:
        for metric in metric_keys:
            s = summary[model_name][f'{cls}_{metric}']
            n = len(s['values'])
            ci = stats.t.ppf(0.975, df=n-1) * s['std'] / sqrt(n)
            print(f'{model_name:<12} {cls:<6} {metric:<14} {s["mean"]:<8.4f} '
                  f'±{s["std"]:<7.4f} [{s["mean"]-ci:.4f}, {s["mean"]+ci:.4f}]')
    print()

### 6.8 Salvar resultados consolidados

In [ ]:
# Salvar summary como JSON (convertendo arrays para listas)
summary_json = {}
for model_name in model_names:
    summary_json[model_name] = {}
    for key, vals in summary[model_name].items():
        summary_json[model_name][key] = {k: (v.tolist() if isinstance(v, np.floating) else v)
                                          for k, v in vals.items()}

with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary_json, f, indent=2)

print(f'Resultados salvos em {RESULTS_DIR}')
print(f'  - summary.json (estatísticas descritivas)')
print(f'  - {{model}}_run{{i}}.json (métricas por execução)')
print(f'  - learning_curves.png')
print(f'  - boxplots.png')
print(f'  - bar_ci.png')